In [84]:
import numpy as np
import pandas as pd

# 과제 10-5장 Linear Regression - 자전거 통행량 예측

시애틀 프리몬트 다리를 지나는 자전거 통행량을 날씨와 계절, 기타 요인에 따라 예측하시오. (p436 참고)

<img src="https://images.seattletimes.com/wp-content/uploads/2020/01/01022020_bike-count_114545.jpg?d=2040x1360" width=400>

Ken Lambert, "Bike ridership hits record highs on 2 Seattle routes", The Seattle Times, Jan. 6, 2020

1. 교재에서 제공하는 FremontBridge.csv 파일과 SeattleWeather.csv 파일을 읽어 pandas 의 DataFrame 을 각각 생성하시오. (https://github.com/jakevdp/bicycle-data)
2. 2020 년 이전 데이터만 선택하시오.
3. 일별 총 자전거 통행량을 계산하고, 그 계산 결과를 'counts' 컬럼에 추가하시오.
4. 월요일~일요일까지 요일을 이진 데이터로 각각 인코딩 하고, 그 결과를 요일을 나타내는 컬럼들(7개)에 추가하시오.
5. 공휴일 여부를 이진 데이터로 인코딩 하고, 그 결과를 'holiday' 컬럼에 추가하시오.
6. 일자별 평균 날씨 (TAVG) 를 계산한 후, 'temp' 컬럼에 추가하시오.
7. 일자별 강수량(PRCP)을 계산한 후, 'rainfall' 컬럼에 추가하시오.
8. 데이터에서 NA 를 제거하시오.
9. LinearRegression 모델을 이용하여 자전거 통행량을 예측하시오.
10. Matplotlib 를 이용하여 예측 결과를 시각화 하시오.


In [76]:
fb = pd.read_csv('https://raw.githubusercontent.com/jakevdp/bicycle-data/refs/heads/main/FremontBridge.csv')
sw = pd.read_csv('https://raw.githubusercontent.com/jakevdp/bicycle-data/refs/heads/main/SeattleWeather.csv')

In [77]:
fb.head()
fb.info()
fb['Date'] = pd.to_datetime(fb['Date'])

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 147278 entries, 0 to 147277
Data columns (total 4 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   Date                          147278 non-null  object 
 1   Fremont Bridge Total          147256 non-null  float64
 2   Fremont Bridge East Sidewalk  147256 non-null  float64
 3   Fremont Bridge West Sidewalk  147255 non-null  float64
dtypes: float64(3), object(1)
memory usage: 4.5+ MB


C:\Users\user\AppData\Local\Temp\ipykernel_24196\1551103630.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  fb['Date'] = pd.to_datetime(fb['Date'])


In [78]:
mask = fb['Date'] < '2020-01-01 00:00:00'
fb = fb[mask]

In [79]:
fb = fb.set_index(keys='Date')
fb.head()

,Fremont Bridge Total,Fremont Bridge East Sidewalk,Fremont Bridge West Sidewalk
Date,,,
2019-11-01 00:00:00,12.0,7.0,5.0
2019-11-01 01:00:00,7.0,0.0,7.0
2019-11-01 02:00:00,1.0,0.0,1.0
2019-11-01 03:00:00,6.0,6.0,0.0
2019-11-01 04:00:00,6.0,5.0,1.0


In [80]:
fb_daily = fb.resample('D').sum()
fb_daily.head()

,Fremont Bridge Total,Fremont Bridge East Sidewalk,Fremont Bridge West Sidewalk
Date,,,
2012-10-03,7042.0,3520.0,3522.0
2012-10-04,6950.0,3416.0,3534.0
2012-10-05,6296.0,3116.0,3180.0
2012-10-06,4012.0,2160.0,1852.0
2012-10-07,4284.0,2382.0,1902.0


In [ ]:
fb_daily.index.day_of_week == 0

# for i in range(7):
#     if fb_daily.day_of_week == 

Index([2, 3, 4, 5, 6, 0, 1, 2, 3, 4,
       ...
       6, 0, 1, 2, 3, 4, 5, 6, 0, 1],
      dtype='int32', name='Date', length=2646)

In [82]:
# weekdays = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
fb['Monday'] = fb.index.day == 'Monday'
fb['Tuesday'] = fb.index.day == 'Tuesday'
fb['Wednesday'] = fb.index.day == 'Wednesday'
fb['Thursday'] = fb.index.day == 'Thursday'
fb['Friday'] = fb.index.day == 'Friday'
fb['Saturday'] = fb.index.day == 'Saturday'
fb['Sunday'] = fb.index.day == 'Sunday'
fb

,Fremont Bridge Total,Fremont Bridge East Sidewalk,Fremont Bridge West Sidewalk,counts,Monday,Tuesday,Wednesday,Thursday,Friday,Saturday,Sunday
Date,,,,,,,,,,,
2019-11-01 00:00:00,12.0,7.0,5.0,24.0,False,False,False,False,False,False,False
2019-11-01 01:00:00,7.0,0.0,7.0,14.0,False,False,False,False,False,False,False
2019-11-01 02:00:00,1.0,0.0,1.0,2.0,False,False,False,False,False,False,False
2019-11-01 03:00:00,6.0,6.0,0.0,12.0,False,False,False,False,False,False,False
2019-11-01 04:00:00,6.0,5.0,1.0,12.0,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...
2019-12-31 19:00:00,19.0,8.0,11.0,38.0,False,False,False,False,False,False,False
2019-12-31 20:00:00,13.0,6.0,7.0,26.0,False,False,False,False,False,False,False
2019-12-31 21:00:00,15.0,8.0,7.0,30.0,False,False,False,False,False,False,False


In [83]:
sw.head()

,STATION,NAME,DATE,AWND,FMTM,PGTM,PRCP,SNOW,SNWD,TAVG,...,WT04,WT05,WT08,WT09,WT13,WT14,WT16,WT17,WT18,WT22
0,USW00024233,"SEATTLE TACOMA AIRPORT, WA US",2012-01-01,10.51,NaN,NaN,0.00,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN
1,USW00024233,"SEATTLE TACOMA AIRPORT, WA US",2012-01-02,10.07,NaN,NaN,0.43,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN
2,USW00024233,"SEATTLE TACOMA AIRPORT, WA US",2012-01-03,5.14,NaN,NaN,0.03,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
3,USW00024233,"SEATTLE TACOMA AIRPORT, WA US",2012-01-04,10.51,NaN,NaN,0.80,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,1.0,NaN,1.0,NaN,NaN,NaN
4,USW00024233,"SEATTLE TACOMA AIRPORT, WA US",2012-01-05,13.65,NaN,NaN,0.05,0.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN
